In [38]:
import pandas as pd

df = pd.read_csv('data/mock/mock_customers.csv')

df.groupby('discount_amount')[['income', 'prior_orders', 'tenure_days']].mean().round(3)

,income,prior_orders,tenure_days
discount_amount,,,
0,-0.291,0.451,187.576
10,0.425,0.581,214.883
20,0.424,0.570,218.948


In [39]:
pd.crosstab(df['season'], df['discount_amount'], normalize='columns').round(3)

discount_amount,0,10,20
season,,,
0,0.255,0.235,0.239
1,0.258,0.242,0.244
2,0.233,0.279,0.277
3,0.254,0.244,0.239


In [40]:
df[['true_ite_10', 'true_ite_20', 'true_ite']].describe().round(4)

,true_ite_10,true_ite_20,true_ite
count,100000.0000,100000.0000,100000.0000
mean,0.0503,0.0400,0.0134
std,0.0385,0.0414,0.0299
min,-0.0301,-0.3903,-0.2470
25%,0.0133,0.0206,0.0000
50%,0.0801,0.0434,0.0000
75%,0.0868,0.0657,0.0146
max,0.1239,0.1740,0.1690


In [34]:
df[['true_ite_10', 'true_ite_20']].corr().round(3)

,true_ite_10,true_ite_20
true_ite_10,1.000,0.426
true_ite_20,0.426,1.000


In [41]:
df.groupby('discount_amount')['purchase_made'].agg(['mean', 'count']).round(4)

,mean,count
discount_amount,,
0,0.0666,59941
10,0.1354,19862
20,0.1304,20197


In [36]:
import pandas as pd

df = pd.read_csv('data/mock/mock_customers.csv')

REQUIRED = ['customer_id', 'discount_amount', 'purchase_made',
            'income', 'prior_orders', 'tenure_days', 'season']

print("Missing columns:", [c for c in REQUIRED if c not in df.columns])
print("Null values:", int(df[REQUIRED].isnull().sum().sum()))
print("Discount levels:", sorted(df['discount_amount'].unique().tolist()))
df['discount_amount'].value_counts(normalize=True).sort_index().round(3)

Missing columns: []
Null values: 0
Discount levels: [0, 10, 20]


discount_amount
0     0.599
10    0.199
20    0.202
Name: proportion, dtype: float64

In [37]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_predict

df = pd.read_csv('data/mock/mock_customers.csv')
CONFOUNDERS = ['income', 'prior_orders', 'tenure_days', 'season']

X = pd.get_dummies(df[CONFOUNDERS], columns=['season'], drop_first=True).astype(float)
T = df['discount_amount'].values

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
proba = cross_val_predict(model, X.values, T,
                          cv=KFold(5, shuffle=True, random_state=42),
                          method='predict_proba')

classes = list(model.fit(X.values, T).classes_)
rows = []
for lvl in [0, 10, 20]:
    p = proba[:, classes.index(lvl)]
    rows.append({'level': lvl, 'min': p.min(), 'max': p.max(),
                 'share_extreme': ((p < 0.02) | (p > 0.98)).mean()})

pd.DataFrame(rows).round(3)

,level,min,max,share_extreme
0,0,0.022,0.986,0.000
1,10,0.007,0.497,0.001
2,20,0.007,0.520,0.001


In [44]:
df = pd.read_csv('data/mock/mock_customers.csv')
(df['true_ite_20'] < -0.07).mean()

In [43]:
treated = df[df['discount_amount'] > 0]
treated.groupby('discount_amount')['true_ite'].describe().round(4)

,count,mean,std,min,25%,50%,75%,max
discount_amount,,,,,,,,
10,19862.0,0.0358,0.0365,-0.0301,0.0087,0.0156,0.0825,0.111
20,20197.0,0.0309,0.0423,-0.2470,0.0099,0.0339,0.0583,0.169


In [45]:
import numpy as np
import pandas as pd

scores = pd.read_csv('data/processed/ite_scores.csv')
raw = pd.read_csv('data/mock/mock_customers.csv')
test = scores[scores['split'] == 'test'].merge(
    raw[['customer_id', 'discount_amount', 'true_ite_10', 'true_ite_20']],
    on='customer_id')

for level in [10, 20]:
    sub = test[test['discount_amount'].isin([0, level])]
    est = sub[f'ite_{level}']
    truth = sub[f'true_ite_{level}']
    print(f"${level} vs $0")
    print(f"  correlation estimated vs design ITE: {np.corrcoef(est, truth)[0, 1]:.3f}")
    print(f"  est std: {est.std():.4f}   design std: {truth.std():.4f}")
    print(f"  est mean: {est.mean():.4f}   design mean: {truth.mean():.4f}")
    print(f"  treated in subset: {(sub['discount_amount'] == level).sum():,}")

$10 vs $0
  correlation estimated vs design ITE: 0.850
  est std: 0.0465   design std: 0.0381
  est mean: 0.0560   design mean: 0.0538
  treated in subset: 3,973
$20 vs $0
  correlation estimated vs design ITE: 0.801
  est std: 0.0364   design std: 0.0402
  est mean: 0.0400   design mean: 0.0421
  treated in subset: 4,039


In [48]:
df.head()

,customer_id,income,prior_orders,tenure_days,season,discount_amount,discount_given,purchase_made,true_ite_10,true_ite_20,true_ite
0,0,0.304717,0.0,443.957261,2,10,1,0,0.016953,-0.015254,0.016953
1,1,-1.039984,5.0,148.219355,0,0,0,1,0.090400,0.120800,0.000000
2,2,0.750451,1.0,280.016861,1,0,0,0,0.012495,0.038777,0.000000
3,3,0.940565,0.0,169.006564,3,0,0,0,0.010594,0.021189,0.000000
4,4,-1.951035,1.0,129.346553,0,0,0,0,0.099510,0.109021,0.000000


In [49]:
import pandas as pd

df = pd.read_csv('data/mock/mock_customers.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values:", int(df.isnull().sum().sum()))
print("\nDiscount amount counts:")
print(df['discount_amount'].value_counts().sort_index())
print("\nRepeat rate:", round(df['purchase_made'].mean(), 4))
print("\nRows per customer_id unique:", df['customer_id'].is_unique)

Shape: (100000, 12)
Columns: ['customer_id', 'income', 'prior_orders', 'tenure_days', 'season', 'device_mobile', 'discount_amount', 'discount_given', 'purchase_made', 'true_ite_10', 'true_ite_20', 'true_ite']

Missing values: 0

Discount amount counts:
discount_amount
0     60036
10    19976
20    19988
Name: count, dtype: int64

Repeat rate: 0.0945

Rows per customer_id unique: True


In [50]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data/mock/mock_customers.csv')

# All observed pre-offer columns (true_ite_* and discount_given excluded)
observed = ['income', 'prior_orders', 'tenure_days', 'season', 'device_mobile']
X = pd.get_dummies(df[observed], columns=['season'], drop_first=True).astype(float)

rows = []
for col in X.columns:
    x = X[[col]].values
    # Association with the discount (treatment)
    t_auc = roc_auc_score(df['discount_given'], x[:, 0]) if X[col].nunique() > 1 else np.nan
    # Association with the outcome
    y_auc = roc_auc_score(df['purchase_made'], x[:, 0]) if X[col].nunique() > 1 else np.nan
    rows.append({'variable': col,
                 'auc_vs_discount': round(t_auc, 3),
                 'auc_vs_outcome': round(y_auc, 3)})

table = pd.DataFrame(rows)
table['distance_from_0.5_discount'] = (table['auc_vs_discount'] - 0.5).abs().round(3)
table['distance_from_0.5_outcome'] = (table['auc_vs_outcome'] - 0.5).abs().round(3)
table

,variable,auc_vs_discount,auc_vs_outcome,distance_from_0.5_discount,distance_from_0.5_outcome
0,income,0.681,0.598,0.181,0.098
1,prior_orders,0.542,0.608,0.042,0.108
2,tenure_days,0.535,0.519,0.035,0.019
3,device_mobile,0.500,0.528,0.000,0.028
4,season_1,0.493,0.513,0.007,0.013
5,season_2,0.524,0.496,0.024,0.004
6,season_3,0.491,0.495,0.009,0.005
